In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.data_processing.urbansound_dataset import UrbanSoundDataset
from src.models.CNN_model import AudioCNN, DeepAudioCNN
#from src.models.mobilenet_model import AudioMobileNetV2

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Обучение на устройство: {device}")

# НОВИТЕ ПЪТИЩА КЪМ СЕГМЕНТИРАНИТЕ ДАННИ!
CSV_PATH = "../data/urbansound_processed.csv"
AUDIO_DIR = "../data/processed_audio/"
MODEL_SAVE_PATH = "../models/audio_CNN.pth"

os.makedirs("../models", exist_ok=True)

Обучение на устройство: mps


In [ ]:
print("Създаване на Dataset обекти...")
train_dataset = UrbanSoundDataset('../data/train_split.csv', '../data/processed_audio')

val_dataset = UrbanSoundDataset('../data/val_split.csv', '../data/processed_audio')
test_dataset = UrbanSoundDataset('../data/test_split.csv', '../data/processed_audio')

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Създаване на Dataset обекти...


AttributeError: 'UrbanSoundDataset' object has no attribute 'label_encoder'

In [ ]:
model = AudioCNN(n_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Scheduler: намалява стъпката наполовина, ако няма подобрение 5 епохи
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

EPOCHS = 100
best_val_loss = float('inf')
patience = 15 # Спираме след 15 епохи без подобрение
patience_counter = 0

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

In [ ]:
for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    current_lr = optimizer.param_groups[0]['lr']
    
    for inputs, labels in tqdm(train_loader, desc=f"Епоха {epoch+1}/{EPOCHS} [LR: {current_lr:.6f}]"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct_train / total_train
    
    # --- VALIDATION ---
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct_val / total_val
    
    # Запазване в историята
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% || Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # --- УМНИ ПРОВЕРКИ ---
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"🌟 Нов най-добър модел е запазен! (Val Loss: {best_val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⚠️ Няма подобрение от {patience_counter} епохи.")
        
    print("-" * 60)
    
    if patience_counter >= patience:
        print(f"🛑 Ранно спиране активирано на епоха {epoch+1}!")
        break

print("🎉 Обучението приключи!")

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='blue')
axes[0].plot(epochs_range, history['val_loss'], label='Validation Loss', color='red')
axes[0].set_title('Промяна на Грешката (Loss)')
axes[0].legend()
axes[0].grid(True, linestyle='--')

axes[1].plot(epochs_range, history['train_acc'], label='Train Accuracy', color='blue')
axes[1].plot(epochs_range, history['val_acc'], label='Validation Accuracy', color='red')
axes[1].set_title('Промяна на Точността (Accuracy)')
axes[1].legend()
axes[1].grid(True, linestyle='--')

plt.show()